In [ ]:
!python3 -m spacy download en_core_web_sm
!pip install indic-nlp-library
from sklearn.model_selection import train_test_split
import pickle
import numpy as np
import re
from tqdm import tqdm
import pandas as pd
import spacy
import json
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
from indicnlp.tokenize import indic_tokenize
nlp = spacy.load("en_core_web_sm")

In [ ]:
# Read JSON data from file
with open('/kaggle/input/devphaseds/train_data1.json', 'r') as file:
    train = json.load(file)

with open('/kaggle/input/devphaseds/val_data1.json', 'r') as file:
    val = json.load(file)
with open('/kaggle/input/testphase-data/test_data1_final.json', 'r') as file:
    test = json.load(file)

In [ ]:
data = {
    "English-Bengali": {
        "train_source": [], "train_target": [], "train_ids": [],
        "val_source": [], "val_ids": []
    },
    "English-Hindi": {
        "train_source": [], "train_target": [], "train_ids": [],
        "val_source": [], "val_ids": []
    }
}

test_data = {
    "English-Bengali": {
        "ids" : [], "sentence" : []
    },
    "English-Hindi": {
        "ids" : [], "sentence" : []
    }
}

In [ ]:
# Reading train DS
for language_pair, language_data in train.items():
    for data_type, data_entries in language_data.items():
        print(f"  Data Type: {data_type}")
        for entry_id, entry_data in data_entries.items():
            data[language_pair]["train_source"].append(entry_data["source"])
            data[language_pair]["train_target"].append(entry_data["target"])
            data[language_pair]["train_ids"].append(entry_id)

#Reading evaluation DS
for language_pair, language_data in val.items():
    for data_type, data_entries in language_data.items():
        print(f"  Data Type: {data_type}")
        for entry_id, entry_data in data_entries.items():
            data[language_pair]["val_source"].append(entry_data["source"])
            data[language_pair]["val_ids"].append(entry_id)

#Reading test DS
for language_pair in test.keys():
    test_data[language_pair]["ids"] = test[language_pair]["Test"].keys()
    test_data[language_pair]["sentence"] = [x["source"] for x in test[language_pair]["Test"].values()]

In [ ]:
tra = 0
for i in data:
    for j in data[i]["val_source"]:
        tra += 1
    print(f'length of {i}: {tra}')

In [ ]:
#Preprocessing English
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner", "tagger", "lemmatizer"])
nlp.max_length = 2_000_000

contraction_patterns = [
    (r"n['’]t", " not"),
    (r"['’]re", " are"),
    (r"['’]s", " is"),
    (r"['’]d", " would"),
    (r"['’]ll", " will"),
    (r"['’]ve", " have"),
    (r"['’]m", " am"),
    (r"\bcan['’]t\b", "cannot"),
    (r"\bwon['’]t\b", "will not"),
]

def expand_contractions(text):
    for pattern, repl in contraction_patterns:
        text = re.sub(pattern, repl, text, flags=re.IGNORECASE)
    return text

def clean_text_for_nmt(text: str) -> str:
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\S+@\S+", "", text)
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    text = re.sub(r"[\r\n\t]", " ", text)
    text = expand_contractions(text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

def preprocess_english_corpus(texts, batch_size=1000, n_process=4):
    cleaned_texts = []
    texts = [clean_text_for_nmt(t) for t in texts]
    for doc in tqdm(nlp.pipe(texts, batch_size=batch_size, n_process=n_process), total=len(texts), desc="Tokenizing"):
        tokens = []
        for token in doc:
            tok = token.text.lower().strip()
            if len(tok) == 0:
                continue
            # Remove leading/trailing hyphens
            tok = tok.strip("-")
            if len(tok) == 0:
                continue
            # Skip pure punctuation, spaces, quotes, currency
            if token.is_punct or token.is_space or token.is_quote or token.is_currency:
                continue
            # Skip junk tokens (like 12-digit numbers or weird alphanumeric)
            if re.match(r"^\d{6,}$", tok):
                continue
            if re.match(r"^[\da-zA-Z]*\d+[a-zA-Z]+[\da-zA-Z]*$", tok):
                continue
            # Keep normal numbers and dates
            if re.match(r"^[\+\-]?\d+(\.\d+)?$", tok):
                tokens.append(tok)
                continue
            # Keep normal words including internal hyphens/apostrophes/dots
            if re.match(r"^[a-zA-Z][a-zA-Z'-.]*[a-zA-Z]$", tok):
                tokens.append(tok)
        cleaned_texts.append(tokens)
    return cleaned_texts

In [ ]:
#Preprocess English train
for i in data:
    for j in data[i]:
        if j=="train_source" or j=="val_source":
            texts = data[i][j]
            texts = preprocess_english_corpus(texts)
            data[i][j] = texts

In [ ]:
#Preprocess English test
for i in test_data:
    texts = test_data[i]["sentence"]
    texts = preprocess_english_corpus(texts)
    test_data[i]["sentence"] = texts

In [ ]:
#Preprocessing Hindi/Bengali
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
from indicnlp.tokenize import indic_tokenize
from tqdm import tqdm
import re

factory = IndicNormalizerFactory()
normalizer_hi = factory.get_normalizer("hi")
normalizer_bn = factory.get_normalizer("bn")

def clean_text_indic(text):
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^\u0900-\u097F\u0980-\u09FF0-9\s]", " ", text)  # keep numbers
    text = re.sub(r"[\r\n\t]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_indic_corpus(texts, lang="hi"):
    cleaned_texts = []
    normalizer = normalizer_hi if lang == "hi" else normalizer_bn
    for text in tqdm(texts, desc=f"Preprocessing {lang.upper()}"):
        text = clean_text_indic(text)
        text = normalizer.normalize(text)
        tokens = indic_tokenize.trivial_tokenize(text, lang)
        filtered_tokens = []
        for tok in tokens:
            tok = tok.strip()
            if len(tok) == 0:
                continue
            if re.match(r"^[\u0900-\u097F\u0980-\u09FF0-9]+$", tok):  # letters/numbers only
                filtered_tokens.append(tok)
        cleaned_texts.append(filtered_tokens)
    return cleaned_texts

In [ ]:
l = {"Hindi":"hi","Bengali":"bn"}
for i in data:
    lan = i.split("-")[1]
    j="train_target"
    texts = data[i][j]
    texts = preprocess_indic_corpus(texts, lang=l[lan])
    data[i][j] = texts

In [ ]:
final_val, final_train, train_data90, val_data10 = {},{},{},{}

for lang in ["Bengali", "Hindi"]:
    pair = f"English-{lang}"
    final_train[f"source_{lang}"] = data[pair]["train_source"]
    final_train[f"target_{lang}"] = data[pair]["train_target"]
    final_train[f"ids_{lang}"] = data[pair]["train_ids"]
    final_val[f"val_{lang}"] = data[pair]["val_source"]
    final_val[f"ids_{lang}"] = data[pair]["val_ids"]
    
    src = final_train[f"source_{lang}"]
    tgt = final_train[f"target_{lang}"]
    ids = final_train[f"ids_{lang}"]

    X_train, X_val, y_train, y_val, ids_train, ids_val = train_test_split(
        src, tgt, ids, train_size=0.9, random_state=42)

    train_data90[f"source_{lang}"] = X_train
    train_data90[f"target_{lang}"] = y_train
    train_data90[f"ids_{lang}"] = ids_train

    val_data10[f"source_{lang}"] = X_val
    val_data10[f"target_{lang}"] = y_val
    val_data10[f"ids_{lang}"] = ids_val


In [ ]:
#Saving preprocessed datasets
for i in ["final_val", "final_train", "train_data90", "val_data10"]:
    with open(f"/kaggle/working/{i}.pkl","wb") as f:
        pickle.dump(globals()[i],f)

for i in test_data:
    for j in ["ids","sentence"]:
        with open(f"/kaggle/working/preprocessed_test{i}_{j}.pkl","wb") as f:
            pickle.dump(list(test_data[f"{i}"][j]), f)